# Low‑RAM Fine‑Tuning of **DeepSeek‑R1‑Distill‑Qwen‑1.5B** on Incident‑Response Logs (macOS M‑series)

This notebook fine‑tunes the 1.5 B‑parameter Qwen model **on a Mac with ~36 GB RAM** by:

* Converting the dataset to fixed‑length padded tensors
* Using **LoRA adapters** (only 0.1 % weights trainable)
* Enable **gradient‑checkpointing**
* Loading the base in **fp16** with `device_map="auto"` (layers spread CPU↔MPS)
* Using **Adafactor** (lighter than Adam)
* Sequence length capped at **256** tokens


In [2]:
!pip install --quiet --upgrade pip
!pip install --quiet transformers datasets peft accelerate scikit-learn torch torchvision


In [3]:
import os, torch, json, pandas as pd
from datasets import Dataset
from torch.utils.data import TensorDataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

DATASET_JSONL = "fine_tuning_dataset.jsonl"   # adjust if your file has a different name
MODEL_NAME    = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
OUTPUT_DIR    = "./deepseek_qwen_lora_ft"
NUM_LABELS    = 2
MAX_LEN       = 256   # truncate/pad length

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)


Running on: mps


In [4]:
# 1) Read JSONL
df = pd.read_json(DATASET_JSONL, lines=True)
ds = Dataset.from_pandas(df, preserve_index=False)

# 2) Train/test split
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, test_ds = split["train"], split["test"]

# 3) Map text labels -> integer
def map_labels(ex):
    return {"labels": 1 if ex["output"].strip().lower() == "malicious" else 0}
train_ds = train_ds.map(map_labels)
test_ds  = test_ds.map(map_labels)

# 4) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    return tokenizer(batch["input"], truncation=True, padding=False, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

# 5) Manual padding to MAX_LEN
pad_id = tokenizer.pad_token_id
def pad(seq, L, val): return seq + [val]*(L-len(seq))
train_ids = torch.tensor([pad(x, MAX_LEN, pad_id) for x in train_ds["input_ids"]], dtype=torch.long)
train_mask= torch.tensor([pad(x, MAX_LEN, 0)     for x in train_ds["attention_mask"]], dtype=torch.long)
train_lab = torch.tensor(train_ds["labels"], dtype=torch.long)

test_ids  = torch.tensor([pad(x, MAX_LEN, pad_id) for x in test_ds["input_ids"]], dtype=torch.long)
test_mask = torch.tensor([pad(x, MAX_LEN, 0)      for x in test_ds["attention_mask"]], dtype=torch.long)
test_lab  = torch.tensor(test_ds["labels"], dtype=torch.long)

train_dataset = TensorDataset(train_ids, train_mask, train_lab)
test_dataset  = TensorDataset(test_ids,  test_mask,  test_lab)

print("TensorDataset shapes:", train_ids.shape, train_mask.shape)


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

TensorDataset shapes: torch.Size([9000, 256]) torch.Size([9000, 256])


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [19]:
# 4️⃣ Load Model & Setup Trainer  (CPU-only, LoRA, low-RAM)

import torch
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# ── 1. Base model: fp32 on CPU (device_map forces everything to CPU) ─────
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    num_labels=NUM_LABELS,
    torch_dtype=torch.float32,
    device_map={"": "cpu"},        # <<< CPU-only
    low_cpu_mem_usage=True,
)

# ── 2. LoRA adapters (tiny trainable set) ────────────────────────────────
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(base_model, lora_cfg)
model.gradient_checkpointing_enable()          # save activation RAM

# ── 3. Ensure pad token ──────────────────────────────────────────────────
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# ── 4. Tuple collator for TensorDataset ──────────────────────────────────
def tuple_tensor_collator(batch):
    ids  = torch.stack([b[0] for b in batch])
    msk  = torch.stack([b[1] for b in batch])
    lbls = torch.stack([b[2] for b in batch])
    return {"input_ids": ids, "attention_mask": msk, "labels": lbls}

# ── 5. Custom Trainer (handles num_items_in_batch) ───────────────────────
class CustomTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch: int | None = None,   # accept & ignore
    ):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── 6. TrainingArguments ────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    run_name="qwen_lora_cls_cpu",
    report_to="none",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,   # effective batch = 4
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    optim="adafactor",               # lighter optimizer; fp32 on CPU
    fp16=False,                      # CPU can’t use fp16 kernels
    remove_unused_columns=False,
)

# ── 7. Build Trainer ────────────────────────────────────────────────────
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,    # TensorDataset from Step 2
    eval_dataset=test_dataset,
    data_collator=tuple_tensor_collator,
    compute_metrics=compute_metrics,
)


Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [21]:
# 5️⃣ Train and Save
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model and tokenizer saved to {OUTPUT_DIR}")

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,0.000000,0.000015,1.000000,0.000000,0.000000,0.000000
1000,0.000000,0.000002,1.000000,0.000000,0.000000,0.000000
1500,0.000000,0.000001,1.000000,0.000000,0.000000,0.000000
2000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2500,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
3000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
3500,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
4000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
4500,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
5000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000


/opt/anaconda3/lib/python3.12/site-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: da558e22-f906-4965-9f4e-087cca00ef6e)') - silently ignoring the lookup for the file config.json in deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/peft/utils/save_and_load.py:236: UserWarning: Could not find a config file in deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B - will assume that the vocabulary was not modified.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pin

✅ Model and tokenizer saved to ./deepseek_qwen_lora_ft
